# 混合精度训练

> 训练跑到第 3000 步，loss 突然变成 NaN，日志里再没有别的线索。混合精度训练最常见的翻车原因之一就藏在这里：FP16 的最小正规格化正数约为 `6.1e-5`，更小的数进入 subnormal 区间后可能下溢为 0，而训练后期的小梯度恰恰常常比这更小。
>
> 降低精度会缩小可表示的数值范围，却也能把可训练的模型规模扩大数倍。如何在「省显存、跑得快」和「数值不崩」之间落位，就是混合精度训练要解决的问题。
> 本附录介绍混合精度训练的四组核心内容：
>
> 1. **浮点表示**：指数位决定表示范围，尾数位决定表示精度，FP16、BF16、FP8 与 FP4 在两者之间取不同配比。
>
> 2. **混合精度机制**：矩阵乘法用低精度提高吞吐，梯度累加、优化器状态与关键归一化保留更高精度。
>
> 3. **稳定手段**：Loss Scaling、FP32 master weight 与 dtype 选择共同解决下溢、累积误差和动态范围问题。
>
> 4. **精度演进**：从 FP16、BF16 到 FP8 的 Fine-Grained Quantization，再到 QLoRA 的 NF4 与 BitNet 的 1.58-bit。


为什么不能把所有计算都换成低精度？把 `0.00001` 累加到 `1.0` 上时，FP16 可能因为尾数位不足而丢失这次变化。但是，同样的低精度用于大规模矩阵乘法时，可以容忍一定舍入误差，尤其配合高精度累加时；误差并不总能相互抵消。

训练因此不会把所有操作统一改成低精度。矩阵乘法可以使用 FP16、BF16 或 FP8 提高吞吐，梯度累加、优化器状态和部分归一化操作则保留 FP32。

这种按操作选择精度的方式称为**混合精度训练**。下面先拆开浮点数的符号位、指数位与尾数位，再解释 Loss Scaling 和 FP32 master weight 分别保护了什么。

## 1. 浮点数的位级结构

我们先问一个基础问题：浮点数在内存里到底怎么存？答案是三段 bit 拼起来：1 个符号位（sign）、若干个指数位（exponent）、剩下的尾数位（mantissa）。这三段各司其职，决定一个数能表示多大、多准。

- 符号位：1 bit，0 正 1 负
- 指数位：决定 dynamic range——能表示的最大值和最小值差几个数量级
- 尾数位：决定 precision——在同一个数量级里，能区分多接近的两个数

那么指数位和尾数位谁更重要？答案是都有用，但总 bit 数固定时，两者互相抢。指数位多，range 大；尾数位多，precision 高。也就是说，在总 bit 数固定的约束下，指数位和尾数位是此消彼长的：多给指数位 1 个，就必须少给尾数位 1 个。

这就是 BF16 设计的出发点。BF16 和 FP16 同样是 16 bit，但把尾数位让出 3 个给指数位，换来更大的 range、更低的 precision。为什么这么做划算？下面用具体数字看。

我们先用一张总表，看常见 dtype 的位分配。

| dtype | bits | 符号 | 指数 | 尾数 | 说明 |
|:---|:---|:---|:---|:---|:---|
| FP32 | 32 | 1 | 8 | 23 | 训练的 master weight / optimizer state 基准 |
| FP16 | 16 | 1 | 5 | 10 | range 小（max≈65504），需要 loss scaling |
| BF16 | 16 | 1 | 8 | 7 | 指数范围接近 FP32，尾数只有 7 bit |
| FP8 E4M3 | 8 | 1 | 4 | 3 | 两种 FP8 格式中精度较高 |
| FP8 E5M2 | 8 | 1 | 5 | 2 | 两种 FP8 格式中范围较大 |
| FP4 E2M1 | 4 | 1 | 2 | 1 | 更低精度，训练需要适配的配方 |
| INT8 | 8 | — | — | — | 整数：需要外部 scale 才能表示浮点 |
| INT4 | 4 | — | — | — | 整数：需要外部 scale 才能表示浮点 |

关键观察：FP16 和 BF16 都是 16 bit，但指数/尾数分配不同；整数格式没有浮点指数与尾数。

换句话说，FP8 的两个变体（E4M3 / E5M2）也是同样的取舍——一个偏精度，一个偏范围。

### 1.1 Dynamic Range 与 Precision

指数位、偏置与保留编码共同决定表示范围。FP16 有 5 个指数位，最大有限值为 65504；BF16 有 8 个指数位，最大有限值约 $3.39\times10^{38}$，接近 FP32 的 $3.40\times10^{38}$。

尾数位决定数值间距。在 1 附近，FP16 的 10 位尾数给出 $2^{-10}\approx0.001$ 的间距；BF16 的 7 位尾数给出 $2^{-7}\approx0.008$，大 8 倍。这是该数量级附近的间距，并非每个数都有固定的相对误差。

梯度可能在 $10^{-4}$ 到 $10^{-7}$，也可能因模型而有其他范围。BF16 可以表示这些数量级。FP16 的最小**正规**正数约 $6.1\times10^{-5}$，但仍能表示直到约 $5.96\times10^{-8}$ 的 subnormal；只是相对精度会下降，再小的值可能舍入到 0，部分内核也可能 flush subnormal。

较粗糙但非零的梯度仍可能比完全归零更有用。BF16 的范围减轻了 FP16 下溢压力，但精度与其他稳定性问题仍需处理。


In [2]:
# 用 torch 的 dtype 实际看看 range 和 precision 的差异
import torch

dtypes = [torch.float32, torch.float16, torch.bfloat16]

print('=== 最大可表示值（dynamic range 上限） ===')
for dt in dtypes:
    # torch.finfo 给出该 dtype 的数值信息
    info = torch.finfo(dt)
    print(f'{str(dt):<20} max = {info.max:<25.6e}  min = {info.min:.6e}')

print()
print('=== 最小正正规数（下溢阈值，比这小就变 0） ===')
for dt in dtypes:
    info = torch.finfo(dt)
    print(f'{str(dt):<20} tiny = {info.tiny:<25.6e}')

print()
print('=== 相对精度（两个相邻数之间的最小相对差距） ===')
for dt in dtypes:
    info = torch.finfo(dt)
    print(f'{str(dt):<20} eps = {info.eps:<25.6e}  (约 {info.eps:.4f})')

print()
print('关键观察：')
print('  FP32 vs BF16 的 max 几乎相同（都到 3.4e+38）——range 一样。')
print('  FP16 的 max 只有 65504——遇到稍大的中间激活就 Inf。')
print('  BF16 的 eps 是 FP16 的 8 倍——precision 差，但训练够用。')

=== 最大可表示值（dynamic range 上限） ===
torch.float32        max = 3.402823e+38               min = -3.402823e+38
torch.float16        max = 6.550400e+04               min = -6.550400e+04
torch.bfloat16       max = 3.389531e+38               min = -3.389531e+38

=== 最小正正规数（下溢阈值，比这小就变 0） ===
torch.float32        tiny = 1.175494e-38             
torch.float16        tiny = 6.103516e-05             
torch.bfloat16       tiny = 1.175494e-38             

=== 相对精度（两个相邻数之间的最小相对差距） ===
torch.float32        eps = 1.192093e-07               (约 0.0000)
torch.float16        eps = 9.765625e-04               (约 0.0010)
torch.bfloat16       eps = 7.812500e-03               (约 0.0078)

关键观察：
  FP32 vs BF16 的 max 几乎相同（都到 3.4e+38）——range 一样。
  FP16 的 max 只有 65504——遇到稍大的中间激活就 Inf。
  BF16 的 eps 是 FP16 的 8 倍——precision 差，但训练够用。


### 1.2 不同精度对 0.1 的表示

「精度损失」听起来抽象，我们直接看 0.1 这个数在不同 dtype 下的实际存储值。0.1 在十进制下是有限小数，二进制展开却无限循环，所以任何有限 bit 的浮点都只能存一个近似值。换句话说，存进去再读出来跟原值差多少，就是精度损失的直观体现。

In [3]:
# 同一个 0.1，在不同精度下存进去再读出来，看偏差
import torch

x = 0.1
print(f'原始值（Python float，本质是 FP64）: {x:.20f}')
print()

for dt in [torch.float32, torch.float16, torch.bfloat16]:
    # cast 过去再 cast 回来，模拟「存进去再读出来」
    stored = torch.tensor(x, dtype=dt).item()
    err = abs(stored - x)
    print(f'{str(dt):<20} 存为 {stored:.20f}  误差 {err:.2e}')

print()
print('再看一个对训练更敏感的场景：小梯度 0.0001')
x = 0.0001
print(f'原始值: {x:.10e}')
print()
for dt in [torch.float32, torch.float16, torch.bfloat16]:
    stored = torch.tensor(x, dtype=dt).item()
    err = abs(stored - x)
    rel_err = err / x if x != 0 else 0
    print(f'{str(dt):<20} 存为 {stored:.10e}  相对误差 {rel_err:.2e}')

print()
print('关键观察：FP16 和 BF16 都还能存住 0.0001（没下溢），但精度不同。')
print('如果梯度再小一档到 1e-7，FP16 的 tiny=6e-5 就兜不住了，会直接变 0。')

原始值（Python float，本质是 FP64）: 0.10000000000000000555

torch.float32        存为 0.10000000149011611938  误差 1.49e-09
torch.float16        存为 0.09997558593750000000  误差 2.44e-05
torch.bfloat16       存为 0.10009765625000000000  误差 9.77e-05

再看一个对训练更敏感的场景：小梯度 0.0001
原始值: 1.0000000000e-04

torch.float32        存为 9.9999997474e-05  相对误差 2.53e-08
torch.float16        存为 1.0001659393e-04  相对误差 1.66e-04
torch.bfloat16       存为 1.0013580322e-04  相对误差 1.36e-03

关键观察：FP16 和 BF16 都还能存住 0.0001（没下溢），但精度不同。
如果梯度再小一档到 1e-7，FP16 的 tiny=6e-5 就兜不住了，会直接变 0。


In [4]:
# 演示 FP16 的下溢：小梯度累加成 0
import torch

# 模拟 gradient accumulation：把 8 个 micro-batch 的梯度相加
# 每个梯度是 1e-7 量级（真实训练里常见）
grad_per_step = torch.full((4,), 1e-7, dtype=torch.float32)

# FP32 累加
acc_fp32 = torch.zeros(4, dtype=torch.float32)
for _ in range(8):
    acc_fp32 += grad_per_step

# FP16 累加（如果梯度本身用 FP16 表示）
acc_fp16 = torch.zeros(4, dtype=torch.float16)
grad_fp16 = grad_per_step.to(torch.float16)
print(f'单个梯度 FP16 表示: {grad_fp16}')
print(f'  （注意：1e-7 < FP16 的 tiny=6e-5，所以直接变成 0）')
for _ in range(8):
    acc_fp16 += grad_fp16

# BF16 累加
acc_bf16 = torch.zeros(4, dtype=torch.bfloat16)
grad_bf16 = grad_per_step.to(torch.bfloat16)
print(f'单个梯度 BF16 表示: {grad_bf16}')
for _ in range(8):
    acc_bf16 += grad_bf16

print()
print(f'FP32 累加结果: {acc_fp32}')
print(f'FP16 累加结果: {acc_fp16}  ← 全是 0，梯度消失')
print(f'BF16 累加结果: {acc_bf16}  ← 也丢精度，但比 FP16 好')
print()
print('这就是为什么训练里的累加（gradient accumulation、optimizer state）必须保 FP32。')

单个梯度 FP16 表示: tensor([1.1921e-07, 1.1921e-07, 1.1921e-07, 1.1921e-07], dtype=torch.float16)
  （注意：1e-7 < FP16 的 tiny=6e-5，所以直接变成 0）
单个梯度 BF16 表示: tensor([1.0012e-07, 1.0012e-07, 1.0012e-07, 1.0012e-07], dtype=torch.bfloat16)

FP32 累加结果: tensor([8.0000e-07, 8.0000e-07, 8.0000e-07, 8.0000e-07])
FP16 累加结果: tensor([9.5367e-07, 9.5367e-07, 9.5367e-07, 9.5367e-07], dtype=torch.float16)  ← 全是 0，梯度消失
BF16 累加结果: tensor([8.0094e-07, 8.0094e-07, 8.0094e-07, 8.0094e-07], dtype=torch.bfloat16)  ← 也丢精度，但比 FP16 好

这就是为什么训练里的累加（gradient accumulation、optimizer state）必须保 FP32。


## 2. FP16 训练

Micikevicius et al. 的 2017 年 Mixed Precision Training 论文解释了两类问题和对应方案：Loss Scaling 与 FP32 master weight。

第一类是**梯度下溢**。FP16 的最小正规正数约 $6\times10^{-5}$；更小的梯度进入 subnormal，精度下降，足够小才变成 0。例如 $10^{-7}$ 是 subnormal，并非必然不可表示。小梯度丢失会损失一部分更新信号。

第二类是**权重更新被舍入掉**。数学上 $1.0+0.0001=1.0001$，但 FP16 会将结果舍回 1.0。这是权重所在数量级附近的尾数精度问题，不能只归因于指数范围。Loss Scaling 和主权重分别处理不同的失效原因。


### 2.1 Loss Scaling

梯度太小会被 FP16 的下限吞掉，这该怎么办？NVIDIA 给出的解法叫做 loss scaling，专门对付这个问题。

它的思路很直接：在 backward 之前，把 loss 乘一个很大的数（比如 $2^{16}$）。换句话说，链式求导出来的梯度也会跟着放大 $2^{16}$ 倍，整体落到 FP16 能表示的范围里。

那么更新权重之前，需要把放大后的梯度除回 $2^{16}$，还原成真实值。公式在代码里是这样写的：

```text
scaled_loss = loss * S
scaled_loss.backward()       # 梯度被放大 S 倍，避开下溢
for p in params:
    p.grad = p.grad / S       # 还原
    p.data -= lr * p.grad     # 用还原后的梯度更新
```

这里的 S 是 loss scaling factor。那么 S 选多大合适？

fixed scaling 用一个固定值（常用 $2^{16}$），简单但脆弱。也就是说，一旦某一步梯度突然变大、scaled 之后超过 65504 就 Inf 了。

dynamic scaling 更稳。每次 backward 后检查梯度里有没有 Inf / NaN，有就减小 S 并跳过本次 optimizer 更新，不会自动重跑当前 batch，没有就维持，连续若干步无异常再把 S 加倍。因此，它动态调整，根据观测到的梯度调整 S。下面用代码演示这两种效果。

In [ ]:
# 用 mini 演示 loss scaling 的效果
import torch

torch.manual_seed(42)
# 模拟一段很小的梯度（真实训练里 deep 层常见）
grad_true = torch.randn(8) * 1e-5  # 量级 1e-5
print(f'真实梯度（FP32）: {grad_true}')
print()

# 不做 scaling，直接 cast 到 FP16
grad_fp16_noscale = grad_true.to(torch.float16)
print(f'直接 cast 到 FP16: {grad_fp16_noscale}')
print(f'  非零元素数: {(grad_fp16_noscale != 0).sum().item()} / 8')
print(f'  → 根据非零数量与误差判断是否发生下溢')

print()
# loss scaling：先放大 S=1024 倍，cast 到 FP16，再除回去
S = 1024
grad_scaled = (grad_true * S).to(torch.float16)
grad_restored = grad_scaled.to(torch.float32) / S
print(f'loss scaling (S={S}):')
print(f'  放大后 cast 到 FP16: {grad_scaled}')
print(f'  除回 S 后还原:      {grad_restored}')
print(f'  与真值最大偏差:     {(grad_true - grad_restored).abs().max().item():.2e}')
print()
print('比较非零数量与还原误差；1e-5 不一定直接下溢为零。')

In [ ]:
# 实现 dynamic loss scaling 的极简版
import torch

class DynamicLossScaler:
    """
    极简 dynamic loss scaler，演示思路。
    真实实现见 torch.amp.GradScaler。
    """
    def __init__(self, init_scale=2**16, growth_factor=2.0, backoff_factor=0.5,
                 growth_interval=2000):
        self.scale = init_scale
        self.growth_factor = growth_factor
        self.backoff_factor = backoff_factor
        self.growth_interval = growth_interval
        self.steps_since_growth = 0

    def scale_loss(self, loss):
        return loss * self.scale

    def unscale_grads(self, grads):
        return [g / self.scale for g in grads]

    def step(self, grads_found_inf):
        """每步 backward 后调用，grads_found_inf 表示是否检测到 Inf/NaN"""
        if grads_found_inf:
            self.scale *= self.backoff_factor
            self.steps_since_growth = 0
            print(f'  发现 Inf，scale 减半到 {self.scale:.1f}')
        else:
            self.steps_since_growth += 1
            if self.steps_since_growth >= self.growth_interval:
                self.scale *= self.growth_factor
                self.steps_since_growth = 0
                print(f'  连续无 Inf，scale 加倍到 {self.scale:.1f}')


# 演示：模拟几步训练，偶尔注入 Inf
scaler = DynamicLossScaler(init_scale=1024)
inf_events = [False, False, True, False, False, False]  # 第 3 步注入 Inf
for step, has_inf in enumerate(inf_events, 1):
    print(f'step {step}: scale={scaler.scale:.1f}', end='')
    scaler.step(has_inf)
print()
print('关键观察：dynamic scaling 在 Inf 时自动回退，无 Inf 时缓慢增长，')
print('始终把 scale 维持在「能用」的边缘——这是它比 fixed scaling 稳的原因。')

### 2.2 保留 FP32 的运算

不应直接把所有 op 都转成 FP16。不同运算对数值精度的要求不同：

- **低精度候选**：matmul、conv 可以使用 Tensor Core，配合更高精度累加。加速比例取决于硬件和工作负载，并非固定 2 倍。
- **敏感运算**：一些 reduction、归一化、Softmax 和 loss 适合内部使用 FP32，因为求和、exp 和 log 会放大数值问题。

PyTorch 的 `torch.autocast` 按 op 选择 dtype，但策略依赖设备与具体运算。CPU 与 CUDA 的列表不同，输出 dtype 也不一定揭示累加精度。可查 [autocast 运算参考](https://docs.pytorch.org/docs/stable/amp.html#autocast-op-reference)。

下面例子先将 logits 舍入为 FP16，再转回 FP32 做 Softmax，隔离观察**输入舍入误差**，并非真正执行 FP16 Softmax。


In [ ]:
# 演示 FP16 舍入 logits 后再做 FP32 Softmax 的误差
import torch

# 一组 logits，其中最大值较大
torch.manual_seed(0)
logits = torch.randn(4) * 20  # 放大数值范围
print(f'logits: {logits}')

# FP32 softmax
probs_fp32 = torch.softmax(logits, dim=-1, dtype=torch.float32)
print(f'FP32 softmax: {probs_fp32}')

# 先将 logits 舍入 FP16，再转回 FP32 做 Softmax
logits_fp16 = logits.to(torch.float16)
probs_fp16_bad = torch.softmax(logits_fp16.float(), dim=-1)
print(f'FP16 舍入输入 + FP32 Softmax: {probs_fp16_bad}')

# 对比误差
err = (probs_fp32 - probs_fp16_bad).abs().max().item()
print(f'最大误差: {err:.2e}')
print()
print('此处 Softmax 在 FP32 计算，比较的是输入舍入误差。')
print('Autocast 的策略依赖后端，应查对应运算列表。')

### 2.3 FP16 训练的常见问题

三个失效原因需要分别看待。

**溢出后的 NaN**：Inf 梯度参与优化器更新可能污染参数。GradScaler 检测非有限梯度后跳过更新；突然非有限的 loss 或参数直方图可帮助定位问题。

**优化器状态的舍入**：Adam 的 momentum 与 variance 长期累积。FP32 状态是常用的稳妥基线，低精度状态需要经过验证的优化器方案。

**Attention score 问题**：某些输入与尺度下，$QK^\top$ 可能溢出或过大。检查 Q/K 数量级和 $1/\sqrt{d_h}$ 缩放，使用稳定 Softmax、合适累加精度或支持的 FlashAttention 实现。序列变长本身不保证 score 变大。

这些问题同时涉及范围与精度。BF16 扩大范围，但不能修复所有失效原因。


## 3. BF16 训练

BF16（Brain Float 16）广泛用于 LLM 训练。它与 FP32 一样有 8 个指数位，但只有 7 个尾数位。

最大有限值约 $3.39\times10^{38}$，最小正规正数约 $1.18\times10^{-38}$，指数范围接近 FP32；尾数更少，精度更粗，最大有限值也略有不同。

常见的 0.01–1.0 权重与 $10^{-4}$–$10^{-7}$ 梯度位于此范围内，因此 BF16 通常不需要 Loss Scaling。

敏感运算与权重更新仍应使用合适的较高精度。BF16 减轻 FP16 的溢出、下溢压力，但不稳定运算、数据或优化仍可能产生 Inf/NaN。给某个模型归属训练精度时，应查公开报告，不推测未披露的配方。


### 3.1 `torch.autocast` 的 Dtype 选择

上下文管理器按运算选择 dtype，不必手动转换每个 op：

```python
with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
    output = model(inputs)  # 按运算选择 BF16/FP32
```

典型候选包括：

- **低精度**：`Linear`、`Conv2d`、`matmul`、`bmm`。
- **列表中指定的较高精度**：敏感归一化、loss 和部分 reduction。

精确列表依赖后端。例如 CPU Softmax 对 BF16 输入可能返回 BF16，不能直接套用 CUDA 策略。Autocast 按需转换操作数，不改变模型保存的参数 dtype。转换开销应与 matmul 收益一起测量。下面 MiniBlock 展示当前环境中的参数与输出 dtype。


In [ ]:
# 演示 autocast 怎么改变 op 的实际 dtype
# CPU 和 CUDA 都支持 autocast，这里按实际环境选择 device_type
import torch
import torch.nn as nn

class MiniBlock(nn.Module):
    """一个迷你 Transformer block：Linear → softmax → Linear"""
    def __init__(self, d):
        super().__init__()
        self.fc1 = nn.Linear(d, d)
        self.fc2 = nn.Linear(d, d)

    def forward(self, x):
        # fc1 是 matmul，autocast 下走 BF16
        h = self.fc1(x)
        # Softmax dtype 依后端策略，CPU 与 CUDA 不一定相同
        h = torch.softmax(h, dim=-1)
        # fc2 又是 matmul，回到 BF16
        return self.fc2(h)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dev_type = 'cuda' if device == 'cuda' else 'cpu'
torch.manual_seed(42)
block = MiniBlock(64).to(device)
x = torch.randn(2, 8, 64, device=device)

# 不开 autocast：默认 FP32
print('=== 不开 autocast ===')
out = block(x)
print(f'fc1.weight dtype: {block.fc1.weight.dtype}')
print(f'output dtype:     {out.dtype}')

print()
# 开 BF16 autocast
print('=== 开 autocast(bfloat16) ===')
with torch.autocast(device_type=dev_type, dtype=torch.bfloat16):
    out = block(x)
print(f'fc1.weight dtype: {block.fc1.weight.dtype}  ← 权重本身还是 FP32')
print(f'output dtype:     {out.dtype}  ← 输出是 BF16')
print()
print('关键观察：autocast 不会改变模型权重的 dtype（还是 FP32），')
print('它只在 forward 时把权重的 cast 副本喂给 matmul op。')

### 3.2 FP32 Master Weight

低精度前向、反向不要求权重本身也低精度存储。常见方案保留 FP32 参数，让选定运算使用低精度副本：

1. 保留 FP32 master weight。
2. 前向选定运算所需的操作数转为 BF16。
3. 反向按运算的精度策略计算梯度。
4. 参数为 FP32 时，其梯度累积在 FP32 中。
5. 优化器更新 FP32 权重。

为什么避免直接 BF16 更新？在 $w=1.0$ 附近，BF16 间距约 0.008，远大于 0.0001 的更新，舍入可能将小变化丢掉。

反复的小更新很重要：1000 次 0.0001 理应改变权重 0.1，但每次低精度更新可能被舍回原值。这展示的是一种风险，并非所有参数必然停在初始化。

普通 PyTorch autocast 中，可保留 FP32 模型参数，不主动将整个模型转为 BF16。FP32 参数的梯度会在 FP32 中累积；GradScaler 负责 FP16 的缩放与溢出检测，并非把所有梯度转换回 FP32 的机制。其他框架也可能采用不同主权重策略。下面对比 BF16 与 FP32 直接累积。


In [ ]:
# 演示 BF16 master weight 累加 vs FP32 master weight 累加
import torch

# 模拟 1000 步训练，每步梯度 1e-4
n_steps = 1000
lr = 1e-4

# 方案 A：直接用 BF16 当 master weight（错误）
w_bf16 = torch.tensor([1.0], dtype=torch.bfloat16)
for _ in range(n_steps):
    grad = torch.tensor([lr], dtype=torch.bfloat16)
    w_bf16 = w_bf16 - grad  # BF16 累加

# 方案 B：FP32 master weight，BF16 只用于 forward（正确）
w_fp32 = torch.tensor([1.0], dtype=torch.float32)
for _ in range(n_steps):
    grad = torch.tensor([lr], dtype=torch.float32)
    w_fp32 = w_fp32 - grad  # FP32 累加

print(f'初始值: 1.0，训练 {n_steps} 步，每步减 {lr}')
print(f'理论最终值: {1.0 - n_steps * lr:.4f}')
print(f'BF16 master weight: {w_bf16.item():.6f}')
print(f'FP32 master weight: {w_fp32.item():.6f}')
print()
print('关键观察：BF16 master 几乎没动（1e-4 < BF16 的 eps=0.008，更新全丢）。')
print('这个例子说明 FP32 master 可以保留小更新。')

### 3.3 BF16 与 FP32 的 Loss Curve

下面定义 TinyLM，分别训练 **300 步**并比较精度。这是使用随机输入和目标 ID 的小型字符级风格模型，并非仓库 nanoGPT 实际训练。

两条曲线可能接近，用于观察这个玩具设置中的数值影响。随机目标使它成为精度实验，不能证明有效语言学习或最终模型质量。

代码有 CUDA 时用 CUDA，否则用 CPU。运行时间与 BF16 速度依赖设备，原来 1–2 分钟只能作为粗略预期。


In [ ]:
# BF16 vs FP32 训练对比（mini 实验，CPU/GPU 都能跑）
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time

# 极简的 char-level 语言模型，只用 Linear + Embedding + 手写 attention
# 不依赖 nn.TransformerEncoder（不同 PyTorch 版本行为不一致）
class TinyCausalAttention(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        self.n_head = n_head
        self.d_k = d_model // n_head
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)

    def forward(self, x):
        b, t, d = x.shape
        qkv = self.qkv(x).view(b, t, 3, self.n_head, self.d_k)
        q, k, v = qkv.unbind(dim=2)  # each [b, t, n_head, d_k]
        q = q.transpose(1, 2)  # [b, n_head, t, d_k]
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_k)
        # causal mask
        mask = torch.tril(torch.ones(t, t, device=x.device, dtype=torch.bool))
        scores = scores.masked_fill(~mask, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        h = attn @ v  # [b, n_head, t, d_k]
        h = h.transpose(1, 2).contiguous().view(b, t, d)
        return self.out(h)

class TinyBlock(nn.Module):
    def __init__(self, d_model, n_head):
        super().__init__()
        self.attn = TinyCausalAttention(d_model, n_head)
        self.fc1 = nn.Linear(d_model, 4 * d_model)
        self.fc2 = nn.Linear(4 * d_model, d_model)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.fc2(F.gelu(self.fc1(self.ln2(x))))
        return x

class TinyLM(nn.Module):
    def __init__(self, vocab_size=64, d_model=64, n_layer=2, n_head=2):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(128, d_model)
        self.blocks = nn.ModuleList([TinyBlock(d_model, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size)

    def forward(self, x, targets=None):
        b, t = x.shape
        pos = torch.arange(t, device=x.device)
        h = self.tok_emb(x) + self.pos_emb(pos)[None, :, :]
        for blk in self.blocks:
            h = blk(h)
        h = self.ln_f(h)
        logits = self.lm_head(h)
        if targets is None:
            return logits, None
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

torch.manual_seed(42)
vocab_size = 64
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dev_type = 'cuda' if device == 'cuda' else 'cpu'
print(f'运行设备: {device}')

# 随机数据（教学演示，不依赖 shakespeare 语料下载）
def get_batch(batch_size=16, seq_len=32):
    x = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)
    y = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)
    return x, y

def train_one(dtype_label, n_steps=300):
    torch.manual_seed(42)
    model = TinyLM().to(device)
    # FP32 master weight 始终保留；BF16 模式靠 autocast
    optim = torch.optim.AdamW(model.parameters(), lr=3e-4)
    losses = []
    use_amp = dtype_label != 'fp32'
    amp_dtype = torch.bfloat16 if dtype_label == 'bf16' else torch.float16
    t0 = time.time()
    for step in range(n_steps):
        x, y = get_batch()
        optim.zero_grad()
        if use_amp:
            with torch.autocast(device_type=dev_type, dtype=amp_dtype):
                _, loss = model(x, y)
        else:
            _, loss = model(x, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optim.step()
        if step % 30 == 0 or step == n_steps - 1:
            losses.append((step, loss.item()))
    elapsed = time.time() - t0
    return losses, elapsed

print('训练 FP32 ...')
losses_fp32, t_fp32 = train_one('fp32', n_steps=300)
print(f'  完成，耗时 {t_fp32:.1f}s')
print('训练 BF16 (autocast) ...')
losses_bf16, t_bf16 = train_one('bf16', n_steps=300)
print(f'  完成，耗时 {t_bf16:.1f}s')

print()
print(f'{"step":<8}{"FP32 loss":<14}{"BF16 loss":<14}')
for (s, l1), (_, l2) in zip(losses_fp32, losses_bf16):
    print(f'{s:<8}{l1:<14.4f}{l2:<14.4f}')
print()
print(f'FP32 总耗时: {t_fp32:.1f}s, BF16 总耗时: {t_bf16:.1f}s')
print('比较本次随机目标实验的 loss 和时间，速度依赖设备。')

In [ ]:
# 画 loss 曲线对比
import matplotlib.pyplot as plt

steps_fp32 = [s for s, _ in losses_fp32]
vals_fp32 = [l for _, l in losses_fp32]
steps_bf16 = [s for s, _ in losses_bf16]
vals_bf16 = [l for _, l in losses_bf16]

plt.figure(figsize=(8, 4.5))
plt.plot(steps_fp32, vals_fp32, marker='o', label='FP32 (master + compute)', linewidth=2)
plt.plot(steps_bf16, vals_bf16, marker='s', label='BF16 (autocast)', linewidth=2)
plt.xlabel('training step')
plt.ylabel('loss')
plt.title('BF16 vs FP32 training loss (random data)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('此图比较玩具随机目标模型的精度路径。')
print('不能由此实验推断真实语言任务的最终质量。')

## 4. FP8 训练

FP8 进一步减少数据位宽。Hopper GPU 提供原生 FP8 Tensor Core 运算，DeepSeek-V3 报告公开了大规模 FP8 训练配方。

只有 8 bit，E4M3 与 E5M2 的范围和精度都比 BF16 紧。Scaling 将张量数值映射到可表示范围。

Scaling 可以按整张张量，也可以更细。DeepSeek-V3 使用 **per-tile/per-block scaling** 限制 outlier 影响。下面介绍格式、动机与实现；没有匹配的基线时，不把整次训练成本归因于 FP8 一项。


### 4.1 E4M3 与 E5M2

FP8 为什么需要两个标准变体？因为训练时它们各司其职：

| 变体 | 指数位 | 尾数位 | range | precision | 常见混合配方用途 |
|:---|:---|:---|:---|:---|:---|
| E4M3 | 4 | 3 | $\pm 448$ | 较高 | 前向（activation、权重） |
| E5M2 | 5 | 2 | $\pm 57344$ | 较低 | 反向（梯度） |

你可能会问：为什么前向用 E4M3、反向用 E5M2？因为前向的 activation 数值范围相对窄但需要精度（区分不同的特征），E4M3 的 3 位尾数更合适；反向的梯度数值范围跨好几个数量级但不需要那么准（更新方向对噪声宽容），E5M2 的大 range 更合适。

梯度既需要足够范围，也需要足够精度。这是配方选择，并非所有 FP8 训练的强制要求；DeepSeek-V3 借助细粒度缩放，对量化张量统一使用 E4M3。

那么硬件如何支持这种混合方案？NVIDIA H100 的 FP8 Tensor Core 支持这两类格式，让 forward 用 E4M3、backward 用 E5M2 的混合方案直接落地。

In [ ]:
# 看看 FP8 两个变体的 range（PyTorch 2.1+ 支持 float8_e4m3fn / e5m2）
import torch

if hasattr(torch, 'float8_e4m3fn'):
    fp8_dtypes = [
        ('FP8 E4M3', torch.float8_e4m3fn),
        ('FP8 E5M2', torch.float8_e5m2),
    ]
    print('=== FP8 两个变体的数值信息 ===')
    for name, dt in fp8_dtypes:
        info = torch.finfo(dt)
        print(f'{name}: max={info.max:<10.1f}  min={info.min:<10.1f}  '
              f'tiny={info.tiny:<10.2e}  eps={info.eps:.4f}')
else:
    print('当前 PyTorch 不支持 FP8 dtype（需要 2.1+），用表格代替：')
    print('  FP8 E4M3: max=448,    tiny=2^-6=0.0156, eps=0.125')
    print('  FP8 E5M2: max=57344,  tiny=2^-14=6e-5,  eps=0.25')

print()
print('=== 对比 BF16 / FP16 ===')
for dt in [torch.bfloat16, torch.float16]:
    info = torch.finfo(dt)
    print(f'{str(dt):<20}: max={info.max:<12.1f} tiny={info.tiny:<12.2e} eps={info.eps:.4f}')
print()
print('关键观察：FP8 的 range 远小于 BF16（448 vs 3.4e+38）。')
print('FP8 需要适配的 scaling 配方；局部 scale 可减轻 outlier 影响。')
print('也存在 per-tensor scaling 配方，应按分布验证。')

### 4.2 DeepSeek-V3 的 Fine-Grained Quantization

为什么需要细粒度量化？DeepSeek-V3 论文（2024）公开了它们的 FP8 训练方案，核心是 **per-tile / per-block scaling factor**：把 activation 和权重切成小块，每块单独算一个 scale，把 FP8 的有效 range 钉到每块数据的实际范围上。

具体怎么切分？DeepSeek-V3 用两级 scaling：

- **activation**：按 $1 \times 128$ 的 tile 切分（一行 128 个元素一组），每组一个 scale
- **权重**：按 $128 \times 128$ 的 block 切分，每组一个 scale

为什么要这么细？因为 activation 里通常有 outlier——少数元素数值比其他大几十倍。你想想：如果整张 tensor 共用一个 scale，outlier 会让 scale 偏大，所有正常元素都被压成几个 FP8 级别，精度被 outlier 一个人拖垮。

换句话说，per-tile scaling 让每块的 scale 贴合自己这段数据的实际分布，outlier 只影响它所在的那一块，其他块照常用满 FP8 精度。

那么代价是什么？scaling factor 本身要占额外显存（activation 每 128 元素一组，weight 每 $128	imes128$ 元素一组），并且 kernel 实现复杂得多——要能在 FP8 矩阵乘法的同时处理 per-block scale 的反量化。内核需要在合并部分乘积时应用各组 scale。下面代码使用均匀整数量化说明分组动机，并非非均匀 E4M3 格式的精确模拟。

In [ ]:
# 演示 per-tensor vs per-tile scaling 的精度差异
import torch
import numpy as np

torch.manual_seed(0)
# 模拟一段 activation，有少数 outlier
x = torch.randn(128) * 0.5
x[5] = 8.0   # outlier
x[60] = -7.0

# 均匀对称整数量化演示，不精确模拟 E4M3 码本
def quantize_dequantize(x, scale, n_levels=127):
    """对称量化 + 反量化"""
    x_q = torch.round(x / scale).clamp(-n_levels, n_levels)
    return x_q * scale

# 方案 1：per-tensor（整段共用一个 scale）
scale_per_tensor = x.abs().max().item() / 127
x_deq_per_tensor = quantize_dequantize(x, scale_per_tensor)

# 方案 2：per-tile（切成 128 组？这里改成 8 组演示，每组 16 个元素）
# 真实 DeepSeek-V3 是 1x128，这里用更小的 tile 让差异更明显
n_tiles = 8
tile_size = len(x) // n_tiles
x_deq_per_tile = torch.zeros_like(x)
for i in range(n_tiles):
    chunk = x[i*tile_size:(i+1)*tile_size]
    s = chunk.abs().max().item() / 127
    x_deq_per_tile[i*tile_size:(i+1)*tile_size] = quantize_dequantize(chunk, s)

# 对比误差
err_per_tensor = (x - x_deq_per_tensor).abs().mean().item()
err_per_tile = (x - x_deq_per_tile).abs().mean().item()

# 排除 outlier 后看正常元素的误差
mask = torch.ones(128, dtype=torch.bool)
mask[[5, 60]] = False
err_normal_tensor = (x[mask] - x_deq_per_tensor[mask]).abs().mean().item()
err_normal_tile = (x[mask] - x_deq_per_tile[mask]).abs().mean().item()

print(f'整段数据平均误差:')
print(f'  per-tensor: {err_per_tensor:.4f}')
print(f'  per-tile(8 组): {err_per_tile:.4f}')
print(f'排除 outlier 后正常元素平均误差:')
print(f'  per-tensor: {err_normal_tensor:.4f}  ← outlier 拉宽 scale，正常元素精度损失大')
print(f'  per-tile:   {err_normal_tile:.4f}  ← outlier 只影响自己那一段')
print()
print('关键观察：per-tile 让 outlier 的破坏范围局限在它所在的 tile，')
print('其他 tile 可保留更多整数级分辨率，说明局部缩放的动机。')

### 4.3 DeepGEMM 与高精度累加

[DeepGEMM](https://github.com/deepseek-ai/DeepGEMM) 提供高效低精度 GEMM。需要分清输入、累加与输出精度：

- **FP8 输入不等于 BF16 累加**。V3 报告周期性将部分乘积提升到 FP32 寄存器，降低累加误差。BF16 可以是输出 dtype，不能据此判断 accumulator。
- **Scale 在 kernel 内应用**。Scale tensor 仍单独存在，融合应用减少一次整张张量反量化的往返。
- **运行时特化**：JIT 为矩阵 shape 生成内核；编译后端与 API 依赖所选版本。
- **TMA 与 Warp Specialization**：Hopper 上异步传输与分工 warp 重叠访存和计算。

这些手段提高 GEMM 效率。V3 整体训练成本还包含许多系统选择，不能据此声称 FP8 单独带来统一的 50% 整体降本。


### 4.4 FP8 训练证据

需要区分论文证据与读者准备开展的本地实验。

**DeepSeek-V3**（2024，MoE，671B 总参数 / 37B 激活）：

- 选定前向与反向 GEMM 使用细粒度 FP8 缩放。
- 量化张量使用 E4M3，部分累加提升到 FP32。
- 验证实验相对 BF16 的训练 loss 误差低于 0.25%，这是 loss 对比，并非所有 benchmark 差距低于 0.5%。

**Llama 系列微调**：读者可以在支持的设备上用 Transformer Engine 评估 FP8。本附录未提供能够证明显存省 30–40%、吞吐 1.5 倍的 Meta 官方结果，应在复现实验中测量。

**Qwen2.5 / Qwen3**：不能从社区微调配方推断未披露的预训练 dtype，应分别查官方报告。

**硬件**：原生 FP8 依赖 GPU 与框架。Transformer Engine 支持合适的 Ada、Hopper、Blackwell 设备；A100/V100 没有这类原生 FP8 Tensor Core 路径。PyTorch 能构造 FP8 张量，不代表可以加速训练。


## 5. FP4 / INT4 训练

4-bit 训练不同于常规 BF16，尤其要区分「全部模型参数低精度训练」与「冻结量化基座、训练 adapter」。

本节讨论 QLoRA 广泛使用的 4-bit adapter 微调、BitNet 的原生三值权重训练，以及低精度预训练的稳定性和内核挑战。


### 5.1 QLoRA、NF4 与 Double Quantization

QLoRA（Dettmers et al., 2023）将预训练基座以 4-bit 存储并冻结，使用较高精度计算，训练小型 LoRA adapter。论文展示的是**单张 48GB GPU 微调 65B**，并非 24GB。

三个核心技术：

- **NF4（NormalFloat 4-bit）**：为近似正态分布权重设计的非均匀 16 级码本，在零附近分配更多精度，不是普通 INT4。下面对称整数示例的 `[-7,…,7]` 是 15 级；正态分位码本说明 NF4 动机，并非精确生产码本。
- **Double Quantization**：再次量化量化常数，减少 scale 存储。分组较小（例如 64 个权重）时，scale 开销较明显。
- **Paged Optimizer**：用 unified-memory 换页管理优化器内存峰值，并非所有 Adam 状态永久放在 CPU。

QLoRA 已应用于 Llama、Qwen 等模型并与较高精度微调比较。它改造预训练基座，不证明全部基座权重可以直接从零用 4-bit 存储训练。Adapter 与 compute dtype 取决于实现。


In [ ]:
# 演示 NF4 的核心思想：分位点量化 vs 均匀量化
import numpy as np
from scipy.stats import norm

np.random.seed(42)
# 模拟一段正态分布的权重
w = np.random.randn(10000).astype(np.float32)

# 对称整数量化示例：15 个级，[-7, 7]
def int4_quant(w):
    scale = np.max(np.abs(w)) / 7
    q = np.round(w / scale).clip(-7, 7)
    return q * scale

# NF4 动机演示：16 个正态分位级，非精确生产码本
# 分位点 = norm.ppf((i + 0.5) / 16) for i in 0..15，然后归一化到 [-1, 1]
nf4_levels = norm.ppf((np.arange(16) + 0.5) / 16)
nf4_levels = nf4_levels / np.max(np.abs(nf4_levels))  # 归一化到 [-1, 1]

def nf4_quant(w):
    scale = np.max(np.abs(w))  # NF4 假设权重已归一化
    w_norm = w / scale
    # 每个 weight 找最近的 NF4 级
    q = np.zeros_like(w)
    for i, val in enumerate(w_norm):
        idx = np.argmin(np.abs(nf4_levels - val))
        q[i] = nf4_levels[idx]
    return q * scale

w_int4 = int4_quant(w)
w_nf4 = nf4_quant(w)

mae_int4 = np.mean(np.abs(w - w_int4))
mae_nf4 = np.mean(np.abs(w - w_nf4))

print(f'权重分布: 正态(0, 1)，样本数 10000')
print(f'INT4 均匀量化 MAE: {mae_int4:.4f}')
print(f'NF4 分位量化  MAE: {mae_nf4:.4f}  ← 通常更低')
print()
print('关键观察：NF4 的 16 个级集中在 0 附近（正态分布的密度高处），')
print('对典型权重分布更贴合。这就是 QLoRA 用 NF4 而不是 INT4 的原因。')

### 5.2 BitNet 的 1.58-bit 预训练

BitNet b1.58 的线性权重为 $\{-1,0,+1\}$，理想信息位宽为 $\log_2 3\approx1.58$ bit。实际存储还包括打包和 scale 开销。

核心思路：

- 三值权重允许专门内核用加法、减法和跳过零值实现矩阵乘。
- b1.58 配方将 activation 量化为 **8-bit 整数**，并非不量化的 BF16。
- Straight-Through Estimator（STE）为不可导量化提供替代梯度，训练仍保留高精度 latent weight。

2024 年论文比较了相同规模下与 FP16 基线的质量和效率。后续 [BitNet b1.58 2B4T 报告](https://arxiv.org/abs/2504.12285) 发布原生低位宽模型。稳定性、内核和集成仍重要；已有开源模型不代表普遍采用。


### 5.3 低位宽预训练的挑战

QLoRA 展示量化基座上的 adapter 微调，BitNet 展示另一类原生低位宽架构，不能由此推断所有 4-bit 预训练配方相同。

**稳定性**：量化误差可能影响早期训练，带来 loss spike 或改变优化。高精度 warmup 后再切换是可能方案，并非统一必需或充分的修复。

**内核**：BF16 库较成熟，FP8 有 Transformer Engine、DeepGEMM，FP4 支持也在发展。可用性依赖硬件、布局和配方，应查实际支持的运算，而不是以为权重压缩就能启用整个训练循环。

**总显存**：激活和优化器状态可能占大头。权重位宽下降不会自动压缩这些部分，其敏感性与长期累积需要可能要求更高精度，收益取决于完整存储账本。

BF16 是常用基线，FP8/FP4 需要验证过的配方；QLoRA 是 adapter 微调。推理 INT4/INT8 内核与训练内核解决不同问题。


## 6. 混合精度训练的常见问题

前面把精度原理讲完了，这一节为什么存在？是为了把散落在各章的坑集中起来。

具体分两组：通用坑（dense 和 MoE 都会遇到）、MoE 专属坑。每组都按「症状 → 原因 → 排查方向」组织，作为实际训练时的 checklist。

### 6.1 Dense 与 MoE 的通用问题

**1. 梯度消失或下溢**

- 症状：loss 停滞，某些梯度张量大量为零。
- 可能原因：FP16 进入 subnormal 或舍入为 0，也可能是低精度累积丢失小变化。低于 $6\times10^{-5}$ 不一定直接归零。
- 排查：记录 min/max 和非零比例；FP16 使用 Loss Scaling，BF16 检查计算与累加 dtype，不默认扩大 autocast 范围就能修复。

**2. Attention 概率饱和**

- 症状：很多权重接近 0 或 1。
- 可能原因：Q/K 太大、缺少缩放、范围或精度问题。饱和不直接证明溢出，序列长度本身也不是充分原因。
- 排查：检查 score、缩放、mask、稳定 Softmax 与内核累加精度，对比 FP32 或支持的 FlashAttention。

**3. 优化器状态舍入**

- 症状：长期训练后 loss 停滞或反弹。
- 可能原因：低精度反复更新丢失信息。
- 排查：查 state dtype，以 FP32 为稳妥基线，或用验证过的低精度优化器，不随意 cast 状态。

**4. 跨 micro-batch 梯度累积**

- 症状：累积训练不同于等效大 batch。
- 可能原因：低精度累加、loss 归一化差异，或其他 batch 相关行为。
- 排查：检查归一化和累积 buffer dtype。FP32 累积可能有帮助，`zero_grad(set_to_none=True)` 本身不会选择 buffer 精度。


### 6.2 MoE 的特有问题

路由对较小的 score 差异敏感。

**1. 路由精度**：死专家或不稳定分配可能有很多原因，包括 gating score 舍入。在架构支持时将 router score 与概率计算保留 FP32 是实用基线，必要时显式关闭该路径的 autocast 并检查 top-k 分配。这是诊断建议，不是所有 MoE 实现的统一要求。

**2. 辅助 loss 精度**：负载均衡 loss 震荡或无效可能涉及概率和统计的累积。敏感 reduction 使用 FP32、统计使用合适 buffer；expert 次数可以是整数，概率和需要足够精度。

**3. 负载不均**：少数 expert 接收超过 80% token 时，检查分配比例、score、容量和均衡目标。精度可放大近似相等的 score 差异，但不应默认它是唯一原因。按架构调整 aux loss 或独立均衡机制。

**4. 共享与路由专家**：有 shared expert 时应分别检查两条路径。不同 dtype 可能影响数值，但不能直接证明共享专家收到更多路由 token 或学到重复特征。先比较精度策略、输出与梯度，再判断原因。


## 7. 实践指南

把前面的内容落地成一张决策表。两个维度：训练阶段（预训练 / 微调 / 推理）和模型结构（dense / MoE）。

### 7.1 训练精度的选择

```text
预训练（从头训）
├── Dense
│   ├── 支持 FP8 的 GPU：FP32 master + 验证过的 FP8 计算配方
│   └── A100：BF16 autocast，状态保留合适的较高精度
└── MoE
    ├── 验证过的大规模设置：FP8 计算与合适累加
    └── 基线复现：BF16，单独检查 router 精度

微调
├── 全参数：BF16 计算，优化器和 master 使用合适精度
├── LoRA：基座 BF16 计算，adapter/master dtype 按框架选择
└── QLoRA：冻结 NF4 基座，计算与 adapter 使用较高精度

推理
├── Dense：支持的 INT4/INT8 权重，FP16/BF16 激活
└── MoE：支持的量化权重，单独检查路由精度
```

预训练重视优化稳定性，adapter 微调常重视显存，推理平衡吞吐、延迟与质量。这些是起点，不保证每个模型和内核都支持，应在目标环境验证；MoE 路由应单独检查。


### 7.2 显存节省的验证

BF16 autocast 不会将所有存储减半，FP32 参数和 Adam 状态可能不变。下面是一种示意账本，而非所有实现的统一规则：

| 组件 | FP32 基线 | BF16 autocast 示例 | FP8 配方示例 |
|:---|:---|:---|:---|
| master weight | FP32，4B/参数 | FP32，4B | FP32，4B |
| 计算权重副本 | 无 | BF16，可能缓存 | FP8 与 scale，依赖配方 |
| activation | FP32 | 选定张量 BF16 | 选定张量 FP8 与 scale |
| 参数梯度 | FP32 | FP32 参数对应 FP32 梯度 | 依赖配方，需查累加精度 |
| Adam state | 两份 FP32，8B | 两份 FP32，8B | 依赖配方，FP32 为基线 |

收益依赖保存的激活、副本、scale 和峰值。下面代码教学假设激活元素数等于参数量，估算 **20P 对比 18P bytes，节省 10%**，不包括计算副本与 workspace。这不是 CUDA 峰值显存实测，也不能证明统一节省 30–40%。


In [ ]:
# 教学估算：对比「全 FP32 训练」vs「BF16 autocast 训练」的显存估算
import torch
import torch.nn as nn

def estimate_memory(model, mode='fp32'):
    """
    估算两种训练模式的显存占用。
    mode='fp32'：全 FP32 训练（weight/grad/adam/activation 全 FP32）
    mode='bf16'：BF16 autocast（master/grad/adam 仍 FP32，activation 是 BF16）
    """
    n = sum(p.numel() for p in model.parameters())

    # master weight：两种模式都是 FP32
    master_weight = n * 4

    # gradient：两种模式都累加到 FP32
    gradient = n * 4

    # Adam state：两种模式都是 FP32（momentum + variance）
    adam_state = n * 8

    # activation 假设：约等于模型参数量（教学简化）
    if mode == 'fp32':
        activation = n * 4   # FP32 activation
    else:  # bf16
        activation = n * 2   # BF16 activation

    total = master_weight + gradient + adam_state + activation

    label = '全 FP32 训练' if mode == 'fp32' else 'BF16 autocast'
    print(f'\n=== {label}（{n/1e6:.1f}M 参数）===')
    print(f'master weight (FP32):  {master_weight/1e9:.3f} GB')
    print(f'gradient (FP32):       {gradient/1e9:.3f} GB')
    print(f'Adam state (FP32×2):   {adam_state/1e9:.3f} GB')
    print(f'activation:            {activation/1e9:.3f} GB')
    print(f'合计:                  {total/1e9:.3f} GB')
    return total

# 拿一个 100M 参数的模型做估算
class DummyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(1000, 100000)

model = DummyModel()
n = sum(p.numel() for p in model.parameters())
print(f'模型参数量: {n/1e6:.1f}M')

t_fp32 = estimate_memory(model, 'fp32')
t_bf16 = estimate_memory(model, 'bf16')

saved = (1 - t_bf16 / t_fp32) * 100
print(f'\nBF16 相比全 FP32 节省: {saved:.1f}%')
print('关键观察：master weight + gradient + Adam state 三者在本例中保持不变，')
print('低精度训练的显存收益主要来自 activation 部分。')

### 7.3 NaN 排查清单

按下面顺序排查：

1. **Loss Scaling**：FP16 检查非有限梯度和跳步，连续跳步可能是 scale 不合适或计算不稳定；正常训练并不必须偶尔跳步。
2. **梯度**：反向后查每个参数 min/max/mean，定位最早 Inf/NaN 层。解释数值前先 unscale。
3. **Attention**：检查 Q/K、mask 和缩放。很尖的 Softmax 可能使梯度小，大 score 不自动等于梯度爆炸。
4. **LayerNorm**：检查方差、epsilon 和 dtype；分母包含 $\sqrt{\mathrm{variance}+\epsilon}$，并非直接除方差。
5. **Warmup**：早期学习率过大可能使权重不稳定。
6. **数据**：Token ID 应在 vocab_size 内，越界一般产生索引或 device-assert 错误，而非直接生成 NaN。

`torch.autograd.set_detect_anomaly(True)` 可打印导致反向非有限值的前向 op，但会减慢训练，开销取决于任务。


## 附录：FP8 工程细节

前面讲的是 FP8 训练的原理和论文数据。这一节给想真正动手复现的读者一份工程导读：怎么读 DeepGEMM 源码、怎么从零搭一个 FP8 微调环境。

### A3.1 DeepGEMM 源码导读

[DeepGEMM](https://github.com/deepseek-ai/DeepGEMM) 适合学习低精度 GEMM，阅读需要 CUDA 和 GPU 架构基础。先固定 commit；仓库持续演进，历史路径和 API 不保证存在于当前 main。

**第一遍：API 与测试**

1. 读 `deep_gemm/__init__.py` 的 dense/grouped GEMM 导出。
2. 沿 data 与 scale 参数读实现。历史 `deep_gemm/core/__init__.py` 路径依赖版本，需找当前对应代码。
3. 读 `tests/` 下 GEMM 测试，历史 `tests/test_fp8_gemm.py` 名称可能变化。

应能回答：scale 是 per-tensor、per-tile 还是 per-block？Grouped GEMM 与普通 GEMM 输入有何区别？

**第二遍：编译与内核**

4. 找运行时编译入口；`deep_gemm/jit_kernels.py` 是历史布局，当前可能不同。
5. 查 `csrc/` 的 CUDA 头文件和源码，找到对应 GEMM。

重点看：

- **Scale 存储**：data 与 scale 是独立 tensor。权重 `[K,N]` 按 $128\times128$ 分组时，逻辑 scale 网格为 `[ceil(K/128),ceil(N/128)]`，方向、padding、物理布局依赖 API。
- **累加**：区分 Tensor Core 部分累加、提升 FP32 与最终 BF16 输出，不能从输出 dtype 推断 BF16 accumulator。
- **Launch**：block 处理输出 tile（如 $128\times128$），grid 覆盖输出；Hopper 的 TMA 异步加载，分工 warp 重叠加载和计算。

**第三遍：MoE Grouped GEMM**

6. 找 grouped GEMM 接口与测试；历史 `deep_gemm/grouped_gemm.py` 依赖版本。
7. 沿每个 expert 的可变 token 数读对应 kernel。

每个 expert 单独发射小 GEMM 开销较大，Grouped GEMM 将工作合并到较少 launch，按 expert 分组；专家矩阵宽度通常相同，token 维度不同。

**前置知识**：CUDA grid/block/thread、shared memory、warp；Hopper TMA、Warp Specialization；CUTLASS/CuTe 的基本矩阵抽象。构建前检查固定版本的 GPU 与 CUDA 要求。


### A3.2 FP8 训练复现指南

先复现一个支持的 Linear 运算，再尝试 Llama 微调。V3 完整大规模预训练不是单卡复现；微调能装多大模型取决于序列、adapter、切分与完整内存账本。

**环境**

```text
硬件：Transformer Engine 支持的 Ada、Hopper 或 Blackwell GPU
CUDA/cuDNN/PyTorch/Python：匹配固定的 Transformer Engine 版本
显存：测量完整模型、优化器、激活与 workspace
```

查 [安装要求](https://docs.nvidia.com/deeplearning/transformer-engine/installation.html)，不能默认任意 CUDA 12.x 组合都兼容。

**依赖**

```bash
# extras 加引号，避免 shell 展开括号
pip install 'transformer_engine[pytorch]'

# 可选 Attention 库，安装本身不会启用 FP8 Attention
pip install flash-attn --no-build-isolation

# 可选训练框架，需配置 FP8 集成
pip install trl
# 或
pip install megatron-core
```

**最小运算示例（Transformer Engine 2.x API）**

```python
import torch
import transformer_engine.pytorch as te
from transformer_engine.common.recipe import DelayedScaling, Format

model = te.Linear(4096, 4096, params_dtype=torch.float32).cuda()
input = torch.randn(16, 4096, device='cuda', dtype=torch.bfloat16)
recipe = DelayedScaling(
    fp8_format=Format.HYBRID,
    margin=0,
    amax_history_len=16,
    amax_compute_algo='max',
)
with te.autocast(enabled=True, recipe=recipe):
    output = model(input)
    loss = output.float().square().mean()
loss.backward()  # 在 autocast 作用域外
```

这只展示一次前向/反向，不是完整微调循环。旧版使用 `te.fp8_autocast(fp8_recipe=...)`，需查对应版本；这里不是用 `te.Linear(..., fp8=True)` 开启。

Scaling 选项包括 `margin`（越大越保守）、`amax_history_len`（历史窗口）、`amax_compute_algo`（`max` 或 `most_recent`），可用配方与选项依版本。

**验证计算路径**

```python
# 输出 dtype 本身不揭示 GEMM 操作数和累加精度
hook = model.register_forward_hook(
    lambda m, i, o: print(f'{m.__class__.__name__} output dtype: {o.dtype}')
)

# 分析实际训练脚本：
# nsys profile -t cuda,nvtx python train.py
```

查配方诊断与 profiler 的 kernel/操作数细节。自定义内核不一定叫 `cublasH8Gemm`，FP8 GEMM 也可返回 BF16。路径未生效时检查作用域、shape、硬件与库兼容。参见 [官方 FP8 示例](https://docs.nvidia.com/deeplearning/transformer-engine/getting_started/index.html)。


## 小结

- [ ] 写出 FP32、FP16、BF16、FP8 E4M3/E5M2 的指数/尾数位，解释范围与精度。
- [ ] 解释 BF16 通常不需要 FP16 式 Loss Scaling 的原因。
- [ ] 解释 FP32 master 如何保留反复的小更新。
- [ ] 查目标后端的 autocast 策略，不假设统一列表。
- [ ] 区分 E4M3/E5M2 混合配方与细粒度全 E4M3 配方。
- [ ] 区分 GEMM 的 FP8 输入、高精度累加和 BF16 输出。
- [ ] 解释 QLoRA 的 NF4、Double Quantization、Paged Optimizer。
- [ ] 分别诊断 MoE 路由和辅助 loss 的精度。
- [ ] 为训练、微调和推理选择并验证精度方案。


## 作业

为什么需要动手练习？因为精度的细节只有自己算过一次才能记住。下面三道题帮你把核心概念落地。

> 可以用 AI 询问思路、拆步骤、检查方向，但不建议直接让 AI「做完这道题」。

精度的细节只有自己算过一次才能记住。

**作业 1：BF16 master weight 累加验证**

我们需要模拟 BF16 master weight 累加 1000 步会发生什么。给定初始权重 $w = 1.0$，每步梯度 $g = 0.0001$，分别用 BF16 和 FP32 做 1000 步累加 $w = w - g$，对比最终值与理论值。

小提示：BF16 的 eps 约 0.008，而 0.0001 远小于 0.008，因此累加会丢精度。

In [16]:
# 作业 1：BF16 vs FP32 master weight 累加
import torch

n_steps = 1000
grad = 1e-4

# TODO: 用 BF16 累加 n_steps 步
w_bf16 = torch.tensor([1.0], dtype=torch.bfloat16)
for _ in range(n_steps):
    w_bf16 -= grad

# TODO: 用 FP32 累加 n_steps 步
w_fp32 = torch.tensor([1.0], dtype=torch.float32)
for _ in range(n_steps):
    w_fp32 -= grad

# TODO: 计算理论最终值
w_theoretical = 1.0 - n_steps * grad

assert w_bf16 is not None and w_fp32 is not None and w_theoretical is not None
assert abs(w_fp32.item() - w_theoretical) < 1e-4, f'FP32 应该接近理论值，差 {abs(w_fp32.item() - w_theoretical)}'
print(f'理论最终值:   {w_theoretical:.6f}')
print(f'BF16 累加结果: {w_bf16.item():.6f}')
print(f'FP32 累加结果: {w_fp32.item():.6f}')
print(f'BF16 与理论差距: {abs(w_bf16.item() - w_theoretical):.6f}')
print(chr(10004) + ' 作业 1 通过：master weight 必须 FP32 的原因就在这里')

理论最终值:   0.900000
BF16 累加结果: 1.000000
FP32 累加结果: 0.899983
BF16 与理论差距: 0.100000
✔ 作业 1 通过：master weight 必须 FP32 的原因就在这里


**作业 2：局部缩放与量化误差**

为什么需要 per-tile scaling？我们先看一段有 outlier 的数据。给定 128 个元素，其中 3 个 outlier，分别用 per-tensor 和 per-tile（tile_size=16）做 4-bit 对称整数量化，计算排除 outlier 后正常元素的平均误差。

这是局部缩放演示，不实现 E4M3 FP8。

小提示：per-tile 的关键是先 `reshape` 成 `[n_tiles, tile_size]`，对每一行独立算 scale，再量化。

In [17]:
# 作业 2：per-tensor vs per-tile scaling
import torch

torch.manual_seed(42)
x = torch.randn(128) * 0.5
x[5], x[60], x[100] = 8.0, -7.0, 6.5  # 3 个 outlier

def quantize_dequantize(x, scale, n_levels=7):  # 4-bit: levels = 7
    x_q = torch.round(x / scale).clamp(-n_levels, n_levels)
    return x_q * scale

# TODO: per-tensor 量化（整段共用一个 scale）
scale_tensor = x.abs().max() / 7
x_deq_tensor = quantize_dequantize(x, scale_tensor)

# TODO: per-tile 量化（tile_size=16，共 8 个 tile）
tile_size = 16
x_deq_tile = torch.zeros_like(x)
for i in range(x.numel() // tile_size):
    tile = x[i * tile_size:(i + 1) * tile_size]
    scale_tile = tile.abs().max() / 7
    x_deq_tile[i * tile_size:(i + 1) * tile_size] = quantize_dequantize(tile, scale_tile)

assert scale_tensor is not None and x_deq_tensor is not None
mask = torch.ones(128, dtype=torch.bool)
mask[[5, 60, 100]] = False
err_tensor = (x[mask] - x_deq_tensor[mask]).abs().mean().item()
err_tile = (x[mask] - x_deq_tile[mask]).abs().mean().item()

print(f'per-tensor 正常元素 MAE: {err_tensor:.4f}')
print(f'per-tile  正常元素 MAE: {err_tile:.4f}')
assert err_tile < err_tensor, 'per-tile 应该让正常元素的误差更小（outlier 只影响自己那段）'
print(chr(10004) + ' 作业 2 通过：per-tile scaling 是 FP8 训练精度的关键工程手段')

per-tensor 正常元素 MAE: 0.2836
per-tile  正常元素 MAE: 0.1143
✔ 作业 2 通过：per-tile scaling 是 FP8 训练精度的关键工程手段


**作业 3：估算 BF16 训练相对全 FP32 训练省了多少显存**

为什么比较显存占用？因为训练成本直接受显存大小限制。我们比较「全 FP32 训练」（weight + gradient + Adam state + activation 全是 FP32）和「BF16 autocast 训练」（master weight + gradient + Adam state 仍 FP32，只 activation 存 BF16）。给定 1B 参数模型，假设 activation 总量等价于 1B 参数（教学简化）。

此简化账本明确不包括临时计算副本与 workspace。

小提示：全 FP32 训练时所有组件都是 4B 或 8B/参数；BF16 autocast 时 master weight（4B）+ gradient（4B）+ Adam state（8B）不变，但 activation 从 4B/元素降到 2B/元素。

In [18]:
# 作业 3：估算 1B 模型的训练显存
# 约定：这里比较的是「全 FP32 训练」vs「BF16 autocast 训练」vs「FP8 训练」
# 全 FP32 训练：weight、gradient、Adam state 全是 FP32
# BF16/FP8 训练：master weight + gradient + Adam state 仍是 FP32，只省 activation

n_params = 1_000_000_000  # 1B

# 场景 A：全 FP32 训练（weight + gradient + adam 全 FP32）
# 提示：weight/gradient 各 4B/参数，Adam state（momentum+variance）8B/参数
weight_fp32_bytes = 4 * n_params
gradient_fp32_bytes = 4 * n_params
adam_state_bytes = 8 * n_params

# 场景 B：BF16 autocast 训练
# master weight 仍是 FP32，gradient 累加到 FP32，Adam state 仍 FP32
# 唯一差异：activation 在 forward 时存 BF16（2B/元素）而非 FP32（4B/元素）
# 假设 activation 总量等价于 1B 参数（教学简化）
activation_fp32_bytes = 4 * n_params
activation_bf16_bytes = 2 * n_params

assert all(v is not None for v in [
    weight_fp32_bytes, gradient_fp32_bytes, adam_state_bytes,
    activation_fp32_bytes, activation_bf16_bytes
])

# 全 FP32 训练显存：weight + grad + adam + activation 全 FP32
total_fp32 = weight_fp32_bytes + gradient_fp32_bytes + adam_state_bytes + activation_fp32_bytes
# BF16 autocast：master FP32 + grad FP32 + adam FP32 + activation BF16
total_bf16 = weight_fp32_bytes + gradient_fp32_bytes + adam_state_bytes + activation_bf16_bytes

print(f'1B 模型训练显存估算（含 activation）:')
print(f'  全 FP32 训练:           {total_fp32/1e9:.2f} GB')
print(f'  BF16 autocast 训练:     {total_bf16/1e9:.2f} GB（省 {(1-total_bf16/total_fp32)*100:.1f}%）')

assert total_bf16 < total_fp32, 'BF16 应该省 activation 显存'
print(chr(10004) + ' 作业 3 通过：master weight 和 optimizer state 无法压缩，低精度的收益主要来自 activation')

1B 模型训练显存估算（含 activation）:
  全 FP32 训练:           20.00 GB
  BF16 autocast 训练:     18.00 GB（省 10.0%）
✔ 作业 3 通过：master weight 和 optimizer state 无法压缩，低精度的收益主要来自 activation


## 参考资料

- Micikevicius et al., [Mixed Precision Training](https://arxiv.org/abs/1710.03740), 2017 — NVIDIA 的 FP16 mixed precision 训练奠基论文，提出 loss scaling
- Kalamkar et al., [A Study of BF16 for Deep Learning](https://arxiv.org/abs/1905.12322), 2019 — BF16 的系统性研究
- DeepSeek-AI, [DeepSeek-V3 Technical Report](https://arxiv.org/abs/2412.19437), 2024 — FP8 训练的 fine-grained quantization 方案、训练成本数据
- DeepSeek-AI, [DeepGEMM](https://github.com/deepseek-ai/DeepGEMM) — FP8 GEMM 开源库，DeepSeek-V3 训练核心组件
- NVIDIA, [Transformer Engine](https://github.com/NVIDIA/TransformerEngine) — NVIDIA 官方 FP8 训练库
- Micikevicius et al., [FP8 Formats for Deep Learning](https://arxiv.org/abs/2209.05433), 2022 — FP8 E4M3 / E5M2 标准化论文
- Dettmers et al., [QLoRA: Efficient Finetuning of Quantized LLMs](https://arxiv.org/abs/2305.14314), 2023 — NF4 + double quantization + paged optimizer
- Wang et al., [BitNet: Scaling 1-bit Transformers for Large Language Models](https://arxiv.org/abs/2310.11453), 2023 — 原始 BitNet 研究
- Ma et al., [The Era of 1-bit LLMs: All Large Language Models are in 1.58 Bits](https://arxiv.org/abs/2402.17764), 2024 — BitNet b1.58
